# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
!uv pip install -U openai python-dotenv pandas

Resolved 24 packages in 566ms                                        
⠙ Preparing packages... (0/20)                                                  
⠙ Preparing packages... (0/20)------------------     0 B/1.29 MiB            
⠙ Preparing packages... (0/20)------------------ 16.00 KiB/1.29 MiB          
⠙ Preparing packages... (0/20)------------------ 32.00 KiB/1.29 MiB          
⠙ Preparing packages... (0/20)------------------ 48.00 KiB/1.29 MiB          
⠙ Preparing packages... (0/20)------------------ 63.87 KiB/1.29 MiB          
⠙ Preparing packages... (0/20)------------------ 79.87 KiB/1.29 MiB          
⠙ Preparing packages... (0/20)------------------ 95.87 KiB/1.29 MiB          
⠙ Preparing packages... (0/20)------------------ 111.87 KiB/1.29 MiB         
⠙ Preparing packages... (0/20)------------------ 127.87 KiB/1.29 MiB         
⠹ Preparing packages... (19/20)----------------- 143.87 KiB/1.29 MiB         
⠹ Preparing packages... (19/20)----------------- 143.87 KiB/1.29 MiB 

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [12]:
#Functio to call the model.
def return_OAI(user_message, temperature=0):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=temperature,
        )

    return (response.choices[0].message.content)

In [46]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data1 = {'table': ["employees", "stores"],
        'definition': ["employee_ID, employee_age, employee_name, employee_yearsinthecompany, store_ID",
                        "store_ID, store_state, store_city, store_address"] 
}

data2 = {'table': ["circuits", "laps", "drivers"],
        'definition': ["circuit_ID, circuit_name",
                        "lap_ID, circuit_ID, lap_number, lap_time, driver_ID",
                        "driver_ID, driver_name, driver_age"]
}


data3 = {'table': ["games", "teams", "players", "match_events"],
        'definition': ["game_ID, home_team_ID, away_team_ID, game_date, home_score, away_score",
                        "team_ID, team_name",
                        "player_ID, player_name, team_ID, position",
                        "event_ID, game_ID, player_ID, event_type, event_minute"]
}

df = pd.DataFrame(data3)
print(df)

          table                                         definition
0         games  game_ID, home_team_ID, away_team_ID, game_date...
1         teams                                 team_ID, team_name
2       players          player_ID, player_name, team_ID, position
3  match_events  event_ID, game_ID, player_ID, event_type, even...


In [47]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [48]:
print(text_tables)

games: game_ID, home_team_ID, away_team_ID, game_date, home_score, away_score
teams: team_ID, team_name
players: player_ID, player_name, team_ID, position
match_events: event_ID, game_ID, player_ID, event_type, event_minute


In [49]:
question_data1 = 'Who are the 10 oldest employees by age in New York city?'
question_data2 = 'What is the age of the driver with the fastest lap on each circuit?'
question_data3 = 'Who are the top scorers of the championship?'

question_data = question_data3

In [ ]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Question:
{question}
"""

In [51]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question=question_data)
print(return_OAI(pqt1))

{
    "tables": ["players", "games"]
}


In [52]:
pqt2 = prompt_question_tables.format(tables=text_tables,
                                     question=question_data)
print(return_OAI(pqt2,1))

{
  "Tables": ["players", "games"]
}


In [53]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question=question_data)
print(return_OAI(pqt3,2))

{ "tables": ["games", "teams", "players"] }


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

Summary:\
Three prompts with different temperatures.\
The results are the same for the two first schemas.\
The third schema, the football schema, the model consistently missed match_events, the one table that actually stores goal data. The missing table, the fact that higher temperature didn't help, and the JSON casing inconsistency between runs.\
In conclusion, the schema naming matters. Temperature of LLM isn't a fix for reasoning gaps, and LLMs associate by name rather than tracing data relationships.